# Batch Motion-Compensation TIC Pipeline (MedSAM2)

Batch version of the single-case `MedSam2_inference` notebook.

For every row in the tracking CSV it will:
1. Auto-locate the **BMODE**, **CEUS** and **MC_VOI** files in the row's `Data Dir`
   using the `Site-Patient-Visit-Bolus` naming convention.
2. Run the same MedSAM2 / MC / no-MC TIC pipeline.
3. Create a per-case output folder `{Site}-{Patient}-{Visit}-{Bolus}/` under the
   destination directory and write the raw + fitted TIC curves there.
4. Append the fitted lognormal parameters and quality metrics back into the master CSV.

> **Inputs per case:** `*-BMODE.nii`, `*-CEUS.nii`, `*-MC_VOI.nii.gz`
> **Master CSV:** `/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/Motion Compensation Comparison 4 patients(MedSAM2).csv`

## 0. Setup & working directory
Same `os.chdir` step as the original notebook so the `src.*` imports resolve.

In [1]:
%reload_ext autoreload
%autoreload 2

import os
from pathlib import Path

print("Start CWD:", Path().cwd())
# Original notebook lives in a sub-folder; step up one level so `src` is importable.
# Adjust this if you launch the batch notebook from a different location.
if Path("src").exists() is False and Path("../src").exists():
    os.chdir(Path(os.getcwd()).parent)
print("Working CWD:", Path().cwd())

Start CWD: /home/ahmed-el-kaffas/Documents/Github/QuantUS/engines/ceus/CLI-Demos
Working CWD: /home/ahmed-el-kaffas/Documents/Github/QuantUS/engines/ceus


## 1. Configuration

Edit the paths here. `MASTER_CSV` is both read (to know which cases to process)
and written (results are appended to the matching row).

In [2]:
# ── Master tracking CSV (read cases from here, write results back here) ────────
MASTER_CSV = "/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/Motion Compensation Comparison 4 patients(MedSAM2).csv"

# ── Destination root for per-case output folders ──────────────────────────────
# Falls back to each row's "TIC curve save path" column when present; otherwise
# this default is used.
DEST_ROOT = "/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results"

# ── MedSAM2 model ─────────────────────────────────────────────────────────────
MEDSAM2_CHECKPOINT = "/home/ahmed-el-kaffas/Documents/Github/QuantUS/MedSAM2/checkpoints/MedSAM2_MRI_LiverLesion.pt"
MEDSAM2_PATH       = os.path.abspath(os.path.join(os.getcwd(), "..", "MedSAM2"))
MODEL_CFG          = "configs/sam2.1_hiera_t512.yaml"
DEVICE             = "cuda"

# ── Loaders (match the original notebook) ─────────────────────────────────────
SCAN_TYPE = "nifti"
SEG_TYPE  = "nifti"

# Set to a list of case keys e.g. ["UCSD-P05-V01-CE1"] to only run those,
# or leave as None to process every row in the CSV.
ONLY_CASES = None

# If True, skip a case whose row already has an R2 value filled in.
SKIP_IF_DONE = True

# bbox padding used during adaptive axial segmentation
BBOX_PADDING = 4

## 2. Imports & MedSAM2 model (loaded once)

In [3]:
import sys
import re
import glob
import traceback
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # batch: render to file, no GUI windows
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

import torch

# Make MedSAM2 importable
print("MedSAM2 path:", MEDSAM2_PATH)
if MEDSAM2_PATH not in sys.path:
    sys.path.insert(0, MEDSAM2_PATH)

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# Project entrypoints (same as single-case notebook)
from src.entrypoints import scan_loading_step, seg_loading_step

# ── Load the model ONCE and reuse for every case ──────────────────────────────
sam2_model      = build_sam2(MODEL_CFG, MEDSAM2_CHECKPOINT, device=DEVICE)
image_predictor = SAM2ImagePredictor(sam2_model)
print("MedSAM2 model loaded.")

MedSAM2 path: /home/ahmed-el-kaffas/Documents/Github/QuantUS/engines/MedSAM2
MedSAM2 model loaded.


## 3. File discovery

Given a row's `Data Dir`, `Site`, `Patient`, `Visit`, `Bolus`, locate the three input
files. The convention from the screenshot is:

```
{Site}-{Patient}-{Visit}-{Bolus}_<timestamp>_..._BMODE.nii
{Site}-{Patient}-{Visit}-{Bolus}_<timestamp>_..._CEUS.nii
{Site}-{Patient}-{Visit}-{Bolus}-MC_VOI.nii.gz
```

Matching is done case-insensitively and tolerates the trailing-space / `P010`-vs-`P10`
quirks present in the tracking CSV by globbing on the `Site-Patient-Visit-Bolus` stem
and falling back to looser patterns.

In [4]:
def _clean(s):
    """Trim whitespace; keep original casing for building the case key."""
    return str(s).strip()

def make_case_key(site, patient, visit, bolus):
    return f"{_clean(site)}-{_clean(patient)}-{_clean(visit)}-{_clean(bolus)}"

def _find_one(data_dir, stem, suffix_patterns, label):
    """
    Search data_dir (recursively) for a file whose name starts with `stem`
    (case-insensitive) and matches one of the suffix patterns.
    suffix_patterns: list of glob-style endings, tried in order.
    Returns the first match or None.
    """
    data_dir = _clean(data_dir)
    candidates = []
    # gather all files once (recursive so V01 sub-folders etc. are covered)
    all_files = glob.glob(os.path.join(data_dir, "**", "*"), recursive=True)
    stem_lower = stem.lower()
    for pat in suffix_patterns:
        pat_lower = pat.lower()
        for fp in all_files:
            name = os.path.basename(fp).lower()
            if name.startswith(stem_lower) and name.endswith(pat_lower):
                candidates.append(fp)
        if candidates:
            break
    if not candidates:
        return None
    # deterministic: shortest path / first sorted
    candidates = sorted(set(candidates))
    if len(candidates) > 1:
        print(f"    [warn] multiple {label} matches for stem '{stem}', using first:")
        for c in candidates:
            print("        -", os.path.basename(c))
    return candidates[0]

def locate_inputs(row):
    """
    Resolve BMODE / CEUS / MC_VOI absolute paths for a CSV row.
    Returns dict with keys bmode, ceus, seg (any may be None if not found).
    """
    site    = _clean(row["Site"])
    patient = _clean(row["Patient Number"])
    visit   = _clean(row["Visit"])
    bolus   = _clean(row["Bolus"])
    data_dir = _clean(row["Data Dir"])

    stem = f"{site}-{patient}-{visit}-{bolus}"   # e.g. UCSD-P05-V01-CE1

    bmode = _find_one(data_dir, stem, ["_BMODE.nii", "_bmode.nii", "BMODE.nii"], "BMODE")
    ceus  = _find_one(data_dir, stem, ["_CEUS.nii", "_ceus.nii", "CEUS.nii"], "CEUS")
    # MC_VOI: stem-MC_VOI.nii.gz
    seg   = _find_one(data_dir, stem, ["-MC_VOI.nii.gz", "-mc_voi.nii.gz", "MC_VOI.nii.gz"], "MC_VOI")

    # Fallback: if the strict stem failed (e.g. P010 dir is actually P10),
    # retry with a looser stem that drops the patient/visit prefix.
    if bmode is None or ceus is None or seg is None:
        loose = f"{site}-{patient}".replace(" ", "")
        if bmode is None:
            bmode = _find_one(data_dir, "", [f"*{visit}*{bolus}*_BMODE.nii"], "BMODE(loose)")
        if ceus is None:
            ceus = _find_one(data_dir, "", [f"*{visit}*{bolus}*_CEUS.nii"], "CEUS(loose)")
        if seg is None:
            seg = _find_one(data_dir, "", [f"*{visit}*{bolus}*MC_VOI.nii.gz"], "MC_VOI(loose)")

    return {"bmode": bmode, "ceus": ceus, "seg": seg, "stem": stem}

> **Note on the loose fallback:** the `*` patterns above are matched with `fnmatch`,
> so the next cell installs that behaviour into `_find_one`.

In [5]:
import fnmatch

def _find_one(data_dir, stem, suffix_patterns, label):
    """
    Locate a file under data_dir (recursive).
    - If `stem` is non-empty: name must start with stem AND end with one of suffix_patterns.
    - If `stem` is empty: each pattern is treated as a full fnmatch glob on the basename.
    Case-insensitive throughout.
    """
    data_dir = _clean(data_dir)
    if not os.path.isdir(data_dir):
        print(f"    [warn] data dir does not exist: {data_dir}")
        return None
    all_files = glob.glob(os.path.join(data_dir, "**", "*"), recursive=True)
    stem_lower = stem.lower()
    candidates = []
    for pat in suffix_patterns:
        pat_lower = pat.lower()
        for fp in all_files:
            if not os.path.isfile(fp):
                continue
            name = os.path.basename(fp).lower()
            if stem:
                if name.startswith(stem_lower) and name.endswith(pat_lower):
                    candidates.append(fp)
            else:
                if fnmatch.fnmatch(name, pat_lower):
                    candidates.append(fp)
        if candidates:
            break
    if not candidates:
        return None
    candidates = sorted(set(candidates))
    if len(candidates) > 1:
        print(f"    [warn] multiple {label} matches for '{stem or suffix_patterns}', using first")
    return candidates[0]

## 4. Core pipeline functions

These are lifted verbatim from the single-case notebook (the 2-D MedSAM2 runner,
the adaptive-bbox derivation, the 3-D frame builder, TIC computation, lognormal
fitting and quality metrics). They operate on a `(image_data, bmode_image_data,
seg_data)` triple passed in per case so nothing relies on global state.

In [6]:
# ── MedSAM2 2D helpers ────────────────────────────────────────────────────────
def slice_to_rgb(slice_2d):
    """Normalize a 2D slice to uint8 RGB."""
    s_min, s_max = slice_2d.min(), slice_2d.max()
    uint8 = ((slice_2d - s_min) / (s_max - s_min + 1e-8) * 255).astype(np.uint8)
    return np.stack([uint8] * 3, axis=-1)

def run_medsam2_2d(image_predictor, slice_2d, bbox_2d, device=DEVICE):
    """Run SAM2ImagePredictor on one 2D slice. Returns (H,W) bool mask or None."""
    x_min, y_min, x_max, y_max = bbox_2d
    if (x_max - x_min) < 2 or (y_max - y_min) < 2:
        return None
    rgb = slice_to_rgb(slice_2d)
    with torch.inference_mode(), torch.autocast(device, dtype=torch.bfloat16):
        image_predictor.set_image(rgb)
        masks, _, _ = image_predictor.predict(
            point_coords=None,
            point_labels=None,
            box=np.array(bbox_2d, dtype=np.float32)[None, :],
            multimask_output=False,
        )
    return masks[0].astype(bool)

def get_adaptive_bbox_at_z(coronal_mask, sagittal_mask, z_abs, bbox, volume_shape, padding=4):
    """Derive a tight 2D axial bbox at absolute slice z_abs from the guidance planes."""
    coronal_col  = coronal_mask[:, z_abs]
    x_active     = np.where(coronal_col)[0]
    sagittal_col = sagittal_mask[:, z_abs]
    y_active     = np.where(sagittal_col)[0]
    if len(x_active) == 0 or len(y_active) == 0:
        return None
    x_min = max(0,               x_active.min() - padding)
    x_max = min(volume_shape[0], x_active.max() + padding)
    y_min = max(0,               y_active.min() - padding)
    y_max = min(volume_shape[1], y_active.max() + padding)
    return [x_min, y_min, x_max, y_max]

In [7]:
# ── 3D MedSAM2 mask for a single frame ────────────────────────────────────────
def compute_sam2_mask_for_frame(frame_idx, image_data, bmode_image_data, seg_data,
                                padding=BBOX_PADDING):
    """Returns dict with ceus volume, mc_mask, sam2_mask for one frame."""
    bbox    = seg_data.motion_compensation.tracked_bboxes[frame_idx]
    volume  = bmode_image_data.pixel_data[:, :, :, frame_idx]            # (X,Y,Z)
    mc_mask = seg_data.motion_compensation.apply_to_mask(
                  seg_data.seg_mask, frame_idx, 0)                       # (X,Y,Z)

    # slice centres from MC mask
    z_mid = int(np.argmax(mc_mask.sum(axis=(0, 1))))
    y_mid = int(np.argmax(mc_mask.sum(axis=(0, 2))))
    x_mid = int(np.argmax(mc_mask.sum(axis=(1, 2))))

    # coronal guidance
    coronal_slice = volume[:, y_mid, :]
    coronal_bbox  = np.array([bbox.z_min, bbox.x_min, bbox.z_max, bbox.x_max], dtype=np.float32)
    coronal_mask  = run_medsam2_2d(image_predictor, coronal_slice, coronal_bbox)

    # sagittal guidance
    sagittal_slice = volume[x_mid, :, :]
    sagittal_bbox  = np.array([bbox.z_min, bbox.y_min, bbox.z_max, bbox.y_max], dtype=np.float32)
    sagittal_mask  = run_medsam2_2d(image_predictor, sagittal_slice, sagittal_bbox)

    mask_3d = np.zeros(volume.shape, dtype=np.uint8)
    if coronal_mask is not None and sagittal_mask is not None:
        for abs_z in range(int(bbox.z_min), int(bbox.z_max)):
            adaptive_bbox = get_adaptive_bbox_at_z(
                coronal_mask, sagittal_mask, abs_z, bbox, volume.shape, padding=padding)
            if adaptive_bbox is None:
                continue
            axial_slice = volume[:, :, abs_z].T
            pred_yx     = run_medsam2_2d(image_predictor, axial_slice, adaptive_bbox)
            if pred_yx is not None:
                mask_3d[:, :, abs_z] = pred_yx.T

    if hasattr(image_data, "intensities_for_analysis"):
        ceus_volume = image_data.intensities_for_analysis[:, :, :, frame_idx]
    else:
        ceus_volume = image_data.pixel_data[:, :, :, frame_idx]

    return {"volume": ceus_volume, "mc_mask": mc_mask, "sam2_mask": mask_3d}

In [8]:
# ── TIC + volume helpers ──────────────────────────────────────────────────────
def compute_tic_from_mask(mask_xyz, volume_xyz):
    voxels = volume_xyz[mask_xyz > 0]
    if len(voxels) == 0:
        return np.nan
    return float(np.mean(voxels))

def compute_volume_from_mask(mask_xyz, voxel_volume_mm3):
    if np.sum(mask_xyz) == 0:
        return 0.0
    return float(np.sum(mask_xyz) * voxel_volume_mm3)

def compute_all_tics(image_data, bmode_image_data, seg_data,
                     frame_start=None, frame_end=None):
    """Loop over every frame (or a sub-range) and build the raw TICs + a time axis.

    frame_start / frame_end: inclusive/exclusive frame indices (like Python slice).
    Pass None to use the full range.

    NOTE: MedSAM2 mask computation is disabled below — it was the dominant cost
    of this function (a full SAM2 image-encoder pass per axial slice, per frame).
    Only the MC and no-MC TICs/volumes are computed now.
    """
    n_frames    = bmode_image_data.pixel_data.shape[-1]
    START_FRAME = int(frame_start) if frame_start is not None else 0
    END_FRAME   = int(frame_end)   if frame_end   is not None else n_frames
    START_FRAME = max(0, START_FRAME)
    END_FRAME   = min(n_frames, END_FRAME)

    tic_mc, tic_nomc = [], []
    vol_mc, vol_nomc = [], []
    frames_computed = []

    voxel_vol = float(np.prod(image_data.pixdim))

    from tqdm import tqdm
    for frame_idx in tqdm(range(START_FRAME, END_FRAME), desc="  TIC frames", leave=False):
        if hasattr(image_data, "intensities_for_analysis"):
            ceus_volume = image_data.intensities_for_analysis[:, :, :, frame_idx]
        else:
            ceus_volume = image_data.pixel_data[:, :, :, frame_idx]

        # res       = compute_sam2_mask_for_frame(frame_idx, image_data, bmode_image_data, seg_data)
        # sam2_mask = res["sam2_mask"]
        mc_mask   = seg_data.motion_compensation.apply_to_mask(seg_data.seg_mask, frame_idx, 0)
        nomc_mask = seg_data.seg_mask

        # tic_sam2.append(compute_tic_from_mask(sam2_mask, ceus_volume))
        tic_mc.append(compute_tic_from_mask(mc_mask,   ceus_volume))
        tic_nomc.append(compute_tic_from_mask(nomc_mask, ceus_volume))
        # vol_sam2.append(compute_volume_from_mask(sam2_mask, voxel_vol))
        vol_mc.append(compute_volume_from_mask(mc_mask,   voxel_vol))
        vol_nomc.append(compute_volume_from_mask(nomc_mask, voxel_vol))
        frames_computed.append(frame_idx)

    frames = np.array(frames_computed)
    if hasattr(image_data, "frame_rate") and image_data.frame_rate > 0:
        time_axis = frames * image_data.frame_rate
        x_label   = "Time (s)"
    else:
        time_axis = frames.astype(float)
        x_label   = "Frame index"

    def decompress(tic):
        return (np.asarray(tic, dtype=float) * 3e4 / 255.0) + 3e4

    return {
        "frames": frames, "time_axis": time_axis, "x_label": x_label,
        "tic_mc": decompress(tic_mc), "tic_nomc": decompress(tic_nomc),
        "vol_mc": np.array(vol_mc), "vol_nomc": np.array(vol_nomc),
    }

In [9]:
# ── Lognormal fitting ─────────────────────────────────────────────────────────
def bolus_lognormal(x, auc, mu, sigma, t0):
    with np.errstate(divide="ignore", invalid="ignore"):
        shifted = x - t0
        result = (auc / (shifted * sigma * np.sqrt(2 * np.pi))) * \
                 np.exp(-((np.log(shifted) - mu) ** 2) / (2 * sigma ** 2))
        result = np.nan_to_num(result, nan=0.0, posinf=0.0, neginf=0.0)
    return result

def fit_lognormal_curve(time, curve):
    """Returns (auc, pe, tp, mtt, t0, mu, sigma, pe_loc, baseline).
    pe and auc are amplitude above baseline. baseline = amin of input curve."""
    curve = np.array(curve, dtype=float)
    baseline = float(np.amin(curve))
    curve = curve - baseline                   # shift so minimum == 0

    if np.amax(curve) <= 0:
        print("    Curve is constant, cannot normalize.")
        return tuple(np.nan for _ in range(9))
    normalizer = np.amax(curve)
    curve = curve / normalizer                 # normalize to 0-1

    auc_guess   = np.sum(curve) * (time[1] - time[0])
    mu_guess    = np.log(np.argmax(curve) + 1e-8)
    sigma_guess = 0.5
    t0_guess    = time[np.argmax(curve)] * 0.15

    mu_max  = np.log(time[-1]) if time[-1] > 0 else 10.0
    auc_max = (np.sum(curve) * (time[1] - time[0])) * 10.0
    auc_guess = min(auc_guess, auc_max)
    mu_guess  = min(mu_guess,  mu_max)

    try:
        params, _ = curve_fit(
            bolus_lognormal, time, curve,
            p0=(auc_guess, mu_guess, sigma_guess, t0_guess),
            bounds=([0., 0., 0.01, 0.], [auc_max, mu_max, 5.0, time[-1]]),
            method="trf", maxfev=10000)
    except Exception as e:
        print(f"    Error fitting curve: {e}")
        return tuple(np.nan for _ in range(9))

    auc, mu, sigma, t0 = params
    auc = auc * normalizer                     # amplitude above baseline
    mtt = np.exp(mu + sigma**2 / 2)
    tp  = np.exp(mu - sigma**2)

    fitted_curve = bolus_lognormal(time, *params)   # 0-1 normalized space
    pe     = float(np.max(fitted_curve)) * normalizer  # amplitude above baseline
    pe_loc = int(np.argmax(fitted_curve))
    return auc, pe, tp, mtt, t0, mu, sigma, pe_loc, baseline

def reconstruct_fitted_curve(time, outcome):
    auc, pe, tp, mtt, t0, mu, sigma, pe_loc, baseline = outcome
    if np.isnan(auc):
        return np.full_like(time, np.nan, dtype=float)
    # auc is in amplitude-above-baseline space; adding baseline gives original units
    return bolus_lognormal(time, auc, mu, sigma, t0) + baseline

def tic_quality_metrics(time, tic_raw, outcome):
    auc, pe, tp, mtt, t0, mu, sigma, pe_loc, baseline = outcome
    fitted = reconstruct_fitted_curve(time, outcome)   # original TIC units
    valid  = ~np.isnan(fitted)
    ss_res = np.sum((tic_raw[valid] - fitted[valid]) ** 2)
    ss_tot = np.sum((tic_raw[valid] - tic_raw[valid].mean()) ** 2)
    r2     = 1 - ss_res / (ss_tot + 1e-8)
    roughness = np.mean(np.abs(np.diff(tic_raw)))
    baseline_noise = np.std(tic_raw[:5])
    snr = pe / (baseline_noise + 1e-8)         # pe is amplitude above baseline
    return {"r2": r2, "roughness": roughness, "snr": snr,
            "auc": auc, "pe": pe, "tp": tp, "mtt": mtt, "t0": t0}


## 5. Per-case driver

`process_case` ties everything together for one CSV row:
loads the three inputs, computes the TICs, fits each one, writes per-case CSVs +
a comparison plot into `{DEST_ROOT}/{case_key}/`, and returns fit parameters +
quality metrics for **all three methods** (no-MC, MC, MedSAM2) written back to the
master CSV as prefixed columns (`nomc_*`, `mc_*`, `sam2_*`).

> The per-case `*_fit_params.csv` also stores all three methods row-by-row.

In [10]:
def _save_case_outputs(case_dir, case_key, tic_data, fits, metrics):
    """Write raw+fitted TIC curve CSV, a fit-params CSV, and a comparison PNG."""
    os.makedirs(case_dir, exist_ok=True)
    t  = tic_data["time_axis"]

    # fit_sam2 = reconstruct_fitted_curve(t, fits["sam2"])
    fit_mc   = reconstruct_fitted_curve(t, fits["mc"])
    fit_nomc = reconstruct_fitted_curve(t, fits["nomc"])

    # 1) raw + fitted TIC curves (one row per frame)
    tic_curve_df = pd.DataFrame({
        "frame":        tic_data["frames"],
        "time":         t,
        # "tic_sam2_raw": tic_data["tic_sam2"],
        "tic_mc_raw":   tic_data["tic_mc"],
        "tic_nomc_raw": tic_data["tic_nomc"],
        # "tic_sam2_fit": fit_sam2,
        "tic_mc_fit":   fit_mc,
        "tic_nomc_fit": fit_nomc,
        # "vol_sam2_mm3": tic_data["vol_sam2"],
        "vol_mc_mm3":   tic_data["vol_mc"],
        "vol_nomc_mm3": tic_data["vol_nomc"],
    })
    tic_curve_path = os.path.join(case_dir, f"{case_key}_tic_curve.csv")
    tic_curve_df.to_csv(tic_curve_path, index=False)

    # 2) fit params + quality for MC and no-MC methods (SAM2 disabled — not computed)
    plabels = ["auc", "pe", "tp", "mtt", "t0", "mu", "sigma", "pe_loc","baseline"]
    rows = []
    vnomc = tic_data["vol_nomc"][tic_data["vol_nomc"] > 0]
    ref_volume_mm3 = float(vnomc[0]) if vnomc.size else 0.0
    for method in ["mc", "nomc"]:
        outcome = fits[method]
        m = metrics[method]
        row = {"method": method}
        for lab, val in zip(plabels, outcome):
            row[lab] = val
        row["r2"]        = m["r2"]
        row["roughness"] = m["roughness"]
        row["snr"]       = m["snr"]
        row["volume_mm3"] = ref_volume_mm3
        rows.append(row)
    fit_params_df = pd.DataFrame(rows)
    fit_params_path = os.path.join(case_dir, f"{case_key}_fit_params.csv")
    fit_params_df.to_csv(fit_params_path, index=False)

    # 3) comparison plot
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(t, tic_data["tic_mc"],   color="red",   lw=2, alpha=0.4, label="MC raw")
    # ax.plot(t, tic_data["tic_sam2"], color="blue",  lw=2, alpha=0.4, label="MedSAM2 raw")
    ax.plot(t, tic_data["tic_nomc"], color="green", lw=2, alpha=0.4, label="No MC raw")
    ax.plot(t, fit_mc,   color="darkred",   lw=2, ls="--", label="MC fit")
    # ax.plot(t, fit_sam2, color="darkblue",  lw=2, ls="--", label="MedSAM2 fit")
    ax.plot(t, fit_nomc, color="darkgreen", lw=2, ls="--", label="No MC fit")
    ax.set_xlabel(tic_data["x_label"]); ax.set_ylabel("Mean intensity")
    ax.set_title(case_key); ax.legend(loc="upper right"); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plot_path = os.path.join(case_dir, f"{case_key}_tic_plot.png")
    fig.savefig(plot_path, dpi=120); plt.close(fig)

    return {"tic_curve": tic_curve_path, "fit_params": fit_params_path, "plot": plot_path}

In [11]:
def process_case(row, dest_root=DEST_ROOT, frame_start=None, frame_end=None):
    """Full pipeline for a single CSV row. Returns a dict of results + status.

    frame_start / frame_end: restrict TIC computation to this frame window
    (inclusive start, exclusive end). Set to None to use all frames.
    """
    case_key = make_case_key(row["Site"], row["Patient Number"], row["Visit"], row["Bolus"])
    print(f"\n=== {case_key} ===")

    inputs = locate_inputs(row)
    if not all([inputs["bmode"], inputs["ceus"], inputs["seg"]]):
        missing = [k for k in ("bmode", "ceus", "seg") if not inputs[k]]
        print(f"  [SKIP] missing inputs: {missing}")
        print(f"         bmode={inputs['bmode']}")
        print(f"         ceus ={inputs['ceus']}")
        print(f"         seg  ={inputs['seg']}")
        return {"case_key": case_key, "status": f"missing:{','.join(missing)}"}

    print(f"  bmode: {os.path.basename(inputs['bmode'])}")
    print(f"  ceus : {os.path.basename(inputs['ceus'])}")
    print(f"  seg  : {os.path.basename(inputs['seg'])}")

    # ── Load data (same entrypoints as single-case notebook) ──────────────────
    image_data       = scan_loading_step(SCAN_TYPE, inputs["ceus"])
    bmode_image_data = scan_loading_step(SCAN_TYPE, inputs["bmode"])
    seg_data         = seg_loading_step(SEG_TYPE, image_data, inputs["seg"], inputs["ceus"])

    # ── TICs + fits ───────────────────────────────────────────────────────────
    tic_data = compute_all_tics(image_data, bmode_image_data, seg_data,
                               frame_start=frame_start, frame_end=frame_end)
    t = tic_data["time_axis"]

    fits = {
        # "sam2": fit_lognormal_curve(t, tic_data["tic_sam2"]),
        "mc":   fit_lognormal_curve(t, tic_data["tic_mc"]),
        "nomc": fit_lognormal_curve(t, tic_data["tic_nomc"]),
    }
    metrics = {
        # "sam2": tic_quality_metrics(t, tic_data["tic_sam2"], fits["sam2"]),
        "mc":   tic_quality_metrics(t, tic_data["tic_mc"],   fits["mc"]),
        "nomc": tic_quality_metrics(t, tic_data["tic_nomc"], fits["nomc"]),
    }

    # destination: prefer the row's save-path column if present, else DEST_ROOT
    row_dest = _clean(row.get("TIC curve save path", "")) or dest_root
    case_dir = os.path.join(row_dest, case_key)
    paths = _save_case_outputs(case_dir, case_key, tic_data, fits, metrics)
    print(f"  wrote: {case_dir}")

    # free GPU memory between cases
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    def _method_results(prefix, fit_out, met, vol_arr):
        auc, pe, tp, mtt, t0, mu, sigma, pe_loc, baseline = fit_out
        vol_pos = vol_arr[vol_arr > 0]
        vol_mm3 = float(vol_pos[0]) if vol_pos.size else 0.0
        return {
            f"{prefix}_AUC":       auc,
            f"{prefix}_PE":        pe,
            f"{prefix}_TP":        tp,
            f"{prefix}_MTT":       mtt,
            f"{prefix}_T0":        t0,
            f"{prefix}_Mu":        mu,
            f"{prefix}_Sigma":     sigma,
            f"{prefix}_R2":        met["r2"],
            f"{prefix}_roughness": met["roughness"],
            f"{prefix}_snr":       met["snr"],
            f"{prefix}_volume_mm3": vol_mm3,
        }

    result = {
        "case_key": case_key,
        "status":   "ok",
        "case_dir": case_dir,
        "paths":    paths,
    }
    result.update(_method_results("nomc", fits["nomc"], metrics["nomc"], tic_data["vol_nomc"]))
    result.update(_method_results("mc",   fits["mc"],   metrics["mc"],   tic_data["vol_mc"]))
    # result.update(_method_results("sam2", fits["sam2"], metrics["sam2"], tic_data["vol_sam2"]))
    return result

## 6. Batch run

Loads the master CSV, iterates over the selected rows, runs each case, and writes
the fitted parameters / quality metrics back into the matching row. The master CSV
is saved after **every** case so a crash mid-batch still preserves completed work.

In [ ]:
# Load master CSV
df = pd.read_csv(MASTER_CSV)
# normalise the Site column (strip stray trailing spaces seen in the file)
df["Site"] = df["Site"].astype(str).str.strip()
print(f"Loaded {len(df)} rows from master CSV")

RESULT_COLS = [
    "nomc_AUC", "nomc_PE", "nomc_TP", "nomc_MTT", "nomc_T0", "nomc_Mu", "nomc_Sigma",
    "nomc_R2", "nomc_roughness", "nomc_snr", "nomc_volume_mm3",
    "mc_AUC",   "mc_PE",   "mc_TP",   "mc_MTT",   "mc_T0",   "mc_Mu",   "mc_Sigma",
    "mc_R2",   "mc_roughness",   "mc_snr",   "mc_volume_mm3",
    # "sam2_AUC", "sam2_PE", "sam2_TP", "sam2_MTT", "sam2_T0", "sam2_Mu", "sam2_Sigma",
    # "sam2_R2", "sam2_roughness", "sam2_snr", "sam2_volume_mm3",
]
for col in RESULT_COLS:
    if col not in df.columns:
        df[col] = np.nan

In [ ]:
summary = []

for idx, row in df.iterrows():
    case_key = make_case_key(row["Site"], row["Patient Number"], row["Visit"], row["Bolus"])

    if ONLY_CASES is not None and case_key not in ONLY_CASES:
        continue

    if SKIP_IF_DONE and "sam2_R2" in df.columns and pd.notna(row.get("sam2_R2", np.nan)):
        print(f"[skip-done] {case_key}")
        summary.append({"case_key": case_key, "status": "skipped-done"})
        continue

    try:
        result = process_case(row)
    except Exception as e:
        print(f"  [ERROR] {case_key}: {e}")
        traceback.print_exc()
        result = {"case_key": case_key, "status": f"error:{e}"}

    summary.append(result)

    # write results back into the matching row & persist immediately
    if result.get("status") == "ok":
        for col in RESULT_COLS:
            df.at[idx, col] = result[col]
        df.to_csv(MASTER_CSV, index=False)
        print(f"  master CSV updated for {case_key}")

print("\nBatch complete.")

## 7. Run summary

In [ ]:
summary_df = pd.DataFrame(summary)
if not summary_df.empty:
    print(summary_df["status"].value_counts().to_string())
summary_df

## 8. (Optional) Re-run a single case with frame-range selection

Use the config cell below to pick the case (by CSV row index or case key) and
optionally restrict the TIC analysis to a **good-frame window** — useful when
the acquisition contains bad frames (motion artefacts, dropout, etc.).

| Variable | Meaning |
|---|---|
| `RERUN_ROW_IDX` | Integer index into the master CSV (`df.iloc[N]`). |
| `FRAME_START` | First frame to include (0-based, inclusive). `None` = start of acquisition. |
| `FRAME_END` | First frame to **exclude** (exclusive). `None` = end of acquisition. |

Results are saved to the same per-case folder and the master CSV is updated.

In [14]:
df

,Site,Patient Number,Visit,Bolus,Data Dir,TIC curve save path,nomc_AUC,nomc_PE,nomc_TP,nomc_MTT,...,sam2_PE,sam2_TP,sam2_MTT,sam2_T0,sam2_Mu,sam2_Sigma,sam2_R2,sam2_roughness,sam2_snr,sam2_volume_mm3
0,UCSD,P05,V01,CE1,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,2.347719e+06,8327.633158,47.536914,383.432851,...,8975.495891,52.800761,363.818451,3.789723e+00,5.253279,1.134351,0.966026,202.171913,205.712944,11964.951175
1,UCSD,P05,V01,CE2,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,2.692861e+06,9435.872726,51.554988,369.244355,...,9978.216332,57.343038,350.113502,1.506493e+00,5.255189,1.098243,0.943186,191.578204,110.398935,12876.443770
2,UCSD,P05,V02,CE1,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,1.712319e+06,6166.061616,40.766837,421.194098,...,5768.764736,73.810720,313.022869,7.473880e-01,5.264685,0.981418,0.870615,296.424250,50.886404,9688.391630
3,UCSD,P05,V02,CE2,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,1.545486e+06,5355.496610,58.746114,344.917507,...,4475.074631,69.547323,317.003898,6.665360e-11,5.253279,1.005620,0.878154,196.537790,41.650516,7650.166766
4,UCSD,P05,V03,CE1,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,2.199110e+06,7795.389946,46.169041,392.423915,...,8458.887316,42.558933,408.729072,2.770115e+00,5.258998,1.228051,0.927537,201.908952,48.378266,6361.262661
5,UCSD,P05,V03,CE2,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,2.186378e+06,7672.627407,46.799319,395.343219,...,8463.288033,40.581516,424.551058,2.700113e+00,5.268459,1.251058,0.944508,187.403573,62.164648,8656.066537
6,UCSD,P07,V01,CE1RUN2,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,2.516931e+06,9404.462362,30.497481,509.045741,...,9832.165646,34.977233,475.330919,1.238431e+01,5.294240,1.318917,0.969462,223.521771,99.747365,3925.238109
7,UCSD,P07,V01,CE2,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,2.906401e+06,10325.053000,49.860917,369.317238,...,10977.000964,49.908712,369.140355,9.269303e+00,5.244183,1.154984,0.937301,181.017471,62.752625,4601.932955
8,UCSD,P07,V02,CE01RUN2,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,1.806230e+06,7175.955174,27.168558,504.604722,...,7742.182951,26.977668,372.326067,6.229600e-01,5.044850,1.322815,0.960951,188.531246,269.389169,4005.526408
9,UCSD,P07,V02,CE2,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...,2.706491e+06,10040.504852,37.079784,431.932617,...,10452.629547,39.093571,420.660688,3.157327e-11,5.249870,1.258536,0.890329,211.985753,311.489425,5074.992779


In [17]:
# ── Frame-range selection ────────────────────────────────────────────────────
# Row to re-run (0-based index into the master CSV)
RERUN_ROW_IDX = 35

# Restrict TIC analysis to frames [FRAME_START, FRAME_END).
# Set either to None to use the full acquisition boundary.
# Example: skip the first 3 bad frames and stop at frame 90:
#   FRAME_START = 3
#   FRAME_END   = 90
FRAME_START = 0   # e.g. 3
FRAME_END   = 550   # e.g. 90

# ── Load CSV and preview the selected row ────────────────────────────────────
df = pd.read_csv(MASTER_CSV)
row = df.iloc[RERUN_ROW_IDX]

n_frames = None  # will be filled after data load below
print(f"Selected row {RERUN_ROW_IDX}:", make_case_key(
    row["Site"], row["Patient Number"], row["Visit"], row["Bolus"]))
if FRAME_START is not None or FRAME_END is not None:
    print(f"  Frame window: [{FRAME_START}, {FRAME_END})")
else:
    print("  Frame window: full acquisition")
row

Selected row 35: SHC-P09-V01-CE2
  Frame window: [0, 550)


Site                                                                 SHC
Patient Number                                                       P09
Visit                                                                V01
Bolus                                                                CE2
Data Dir               /media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...
TIC curve save path    /media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPU...
nomc_AUC                                                             NaN
nomc_PE                                                              NaN
nomc_TP                                                              NaN
nomc_MTT                                                             NaN
nomc_T0                                                              NaN
nomc_Mu                                                              NaN
nomc_Sigma                                                           NaN
nomc_R2                                            

In [18]:
# Re-run the selected case with the chosen frame window
res = process_case(row, frame_start=FRAME_START, frame_end=FRAME_END)
res


=== SHC-P09-V01-CE2 ===
  bmode: SHC-P09-V01-CE2_18.45.33_mf_sip_capture_50_2_1_0_BMODE.nii
  ceus : SHC-P09-V01-CE2_18.45.33_mf_sip_capture_50_2_1_0_CEUS.nii
  seg  : SHC-P09-V01-CE2-MC_VOI.nii.gz


  wrote: /media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/SHC-P09-V01-CE2


{'case_key': 'SHC-P09-V01-CE2',
 'status': 'ok',
 'case_dir': '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/SHC-P09-V01-CE2',
 'paths': {'tic_curve': '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/SHC-P09-V01-CE2/SHC-P09-V01-CE2_tic_curve.csv',
  'fit_params': '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/SHC-P09-V01-CE2/SHC-P09-V01-CE2_fit_params.csv',
  'plot': '/media/ahmed-el-kaffas/20TB-HDD/Yuanshan/3DMPUS/Motion_compensation_results/SHC-P09-V01-CE2/SHC-P09-V01-CE2_tic_plot.png'},
 'nomc_AUC': np.float64(697250.9859251379),
 'nomc_PE': np.float64(4924.459600470032),
 'nomc_TP': np.float64(25.014646637829237),
 'nomc_MTT': np.float64(186.11461150600275),
 'nomc_T0': np.float64(6.796146689468803),
 'nomc_Mu': np.float64(4.557395622835079),
 'nomc_Sigma': np.float64(1.1566910149239975),
 'nomc_R2': np.float64(0.7278635425450382),
 'nomc_roughness': np.float64(441.42307215482856),
 'nomc_snr': 